# Telecom X — Parte 2: Previsão de Churn

Este notebook usa o dataset **processado na Parte 1** (`data/processed/telecom_transformed.parquet`) para treinar modelos de Machine Learning e prever churn.

**Data:** 05/03/2026

## Pré-requisito
Rode a Parte 1 antes (no seu repositório `telecomx-churn`):

```bash
python scripts/run_etl.py
```

Isso deve gerar:
- `data/processed/telecom_transformed.parquet`
- `data/processed/telecom_transformed.csv`


In [ ]:
# Se estiver no Colab e precisar instalar:
# !pip -q install pandas numpy matplotlib scikit-learn imbalanced-learn joblib


## 1) Carregar dataset tratado (Parte 1)

In [ ]:
import os
import pandas as pd
import numpy as np

PARQUET = "data/processed/telecom_transformed.parquet"
CSV = "data/processed/telecom_transformed.csv"

if os.path.exists(PARQUET):
    df = pd.read_parquet(PARQUET)
    origem = PARQUET
elif os.path.exists(CSV):
    df = pd.read_csv(CSV)
    origem = CSV
else:
    raise FileNotFoundError("Não achei o dataset processado. Rode antes: python scripts/run_etl.py")

print("Carregado:", origem, "| Shape:", df.shape)
display(df.head(5))

## 2) Definir alvo (churn) e remover colunas irrelevantes

O alvo costuma ser `churn` (Yes/No). Se for outro nome, ajuste.

Também removemos colunas do tipo ID (alta cardinalidade) para não atrapalhar o modelo.

In [ ]:
# 2.1) alvo
if "churn" not in df.columns:
    # fallback: tenta achar nome parecido
    candidatos = [c for c in df.columns if "churn" in c.lower() or "evas" in c.lower()]
    if candidatos:
        alvo = candidatos[0]
        print("Usando alvo detectado:", alvo)
    else:
        raise KeyError("Não encontrei coluna de churn. Verifique seu dataset.")
else:
    alvo = "churn"

# 2.2) remove possíveis IDs (heurística)
cols_drop = []
for c in df.columns:
    if c == alvo:
        continue
    nome = c.lower()
    if nome in ["customerid","customer_id","id","cliente_id"] or nome.endswith("id"):
        cols_drop.append(c)
        continue
    nunique = df[c].nunique(dropna=True)
    if nunique / max(len(df),1) > 0.95:
        cols_drop.append(c)

cols_drop = sorted(set(cols_drop))
print("Colunas removidas:", cols_drop if cols_drop else "(nenhuma)")

df_model = df.drop(columns=cols_drop, errors="ignore").copy()

## 3) Proporção de churn (desbalanceamento)

Checamos contagem e proporção para decidir se vale SMOTE/ajustes.

In [ ]:
churn_counts = df_model[alvo].value_counts(dropna=False)
churn_prop = df_model[alvo].value_counts(normalize=True, dropna=False) * 100
display(pd.DataFrame({"contagem": churn_counts, "proporção_%": churn_prop.round(2)}))

## 4) Preparação: X/y, split treino/teste, encoding e escala

Usamos **Pipeline + ColumnTransformer** (evita vazamento e deixa reproduzível).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

X = df_model.drop(columns=[alvo])
y_raw = df_model[alvo].copy()

# converte y para 0/1
if y_raw.dtype == "object":
    mapa = {"yes":1,"no":0,"sim":1,"não":0,"nao":0,"true":1,"false":0,"1":1,"0":0}
    y = y_raw.astype(str).str.strip().str.lower().map(mapa)
    if y.isna().any():
        uniq = list(pd.unique(y_raw.astype(str)))
        if len(uniq) == 2:
            y = y_raw.astype(str).map({uniq[0]:0, uniq[1]:1})
        else:
            raise ValueError(f"Não consegui converter churn para binário. Valores únicos: {uniq[:10]}")
    y = y.astype(int)
else:
    y = y_raw.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess_scale = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

print("Treino:", X_train.shape, "Teste:", X_test.shape)
print("Numéricas:", len(num_cols), " | Categóricas:", len(cat_cols))

## 5) Balanceamento (opcional)

Você pode ligar SMOTE se o churn estiver muito desbalanceado.

> Importante: SMOTE só no treino (dentro do pipeline).

In [ ]:
USAR_SMOTE = False  # mude para True se quiser testar

if USAR_SMOTE:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    smote = SMOTE(random_state=42)
    print("SMOTE ligado ✅")
else:
    smote = None
    ImbPipeline = None
    print("SMOTE desligado (padrão)")

## 6) Treinar modelos e avaliar

Vamos treinar e comparar:
- Regressão Logística
- KNN
- SVM (linear)
- RandomForest

Métricas:
- Precision/Recall/F1
- ROC-AUC
- Matriz de confusão

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

def avaliar(nome, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    print(f"\n=== {nome} ===")
    print("Matriz de confusão:\n", confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=4))

    if hasattr(modelo, "predict_proba"):
        y_score = modelo.predict_proba(X_test)[:, 1]
    else:
        y_score = modelo.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score)
    ap = average_precision_score(y_test, y_score)
    print("ROC-AUC:", round(auc, 4), "| PR-AUC:", round(ap, 4))
    return auc, ap, y_score

modelos = {
    "LogReg": LogisticRegression(max_iter=2500, class_weight="balanced"),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "SVM": SVC(kernel="linear", class_weight="balanced", probability=True),
    "RandomForest": RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1, class_weight="balanced_subsample"),
}

resultados = {}
treinados = {}

for nome, clf in modelos.items():
    prep = preprocess_tree if nome == "RandomForest" else preprocess_scale

    if USAR_SMOTE and smote is not None and nome != "RandomForest":
        pipe = ImbPipeline([("prep", prep), ("smote", smote), ("clf", clf)])
    else:
        pipe = Pipeline([("prep", prep), ("clf", clf)])

    pipe.fit(X_train, y_train)
    auc, ap, y_score = avaliar(nome, pipe, X_test, y_test)
    resultados[nome] = {"roc_auc": auc, "pr_auc": ap}
    treinados[nome] = (pipe, y_score)

display(pd.DataFrame(resultados).T.sort_values("roc_auc", ascending=False))

## 7) Curvas ROC e Precision-Recall

In [ ]:
plt.figure()
for nome, (pipe, y_score) in treinados.items():
    fpr, tpr, _ = roc_curve(y_test, y_score)
    plt.plot(fpr, tpr, label=f"{nome} (AUC={resultados[nome]['roc_auc']:.3f})")
plt.plot([0,1],[0,1], linestyle="--", label="Aleatório")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("Curvas ROC")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure()
for nome, (pipe, y_score) in treinados.items():
    p, r, _ = precision_recall_curve(y_test, y_score)
    plt.plot(r, p, label=f"{nome} (PR-AUC={resultados[nome]['pr_auc']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curvas Precision-Recall")
plt.legend()
plt.tight_layout()
plt.show()

## 8) Importância de variáveis

Usamos `permutation_importance` no melhor modelo (ROC-AUC) para identificar as features mais relevantes.

> Isso é um jeito **compatível** com quase qualquer modelo.

In [ ]:
from sklearn.inspection import permutation_importance

melhor_nome = max(resultados, key=lambda k: resultados[k]["roc_auc"])
melhor_pipe = treinados[melhor_nome][0]
print("Melhor modelo:", melhor_nome, "| ROC-AUC:", resultados[melhor_nome]["roc_auc"])

r = permutation_importance(melhor_pipe, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)

# nomes das features pós one-hot
prep = melhor_pipe.named_steps["prep"]
feat_names = []

feat_names.extend(num_cols)
if len(cat_cols) > 0:
    ohe = prep.named_transformers_["cat"]
    feat_names.extend(list(ohe.get_feature_names_out(cat_cols)))

importancias = pd.Series(r.importances_mean, index=feat_names).sort_values(ascending=False)
display(importancias.head(25).to_frame("importância_permuta"))

plt.figure(figsize=(8,6))
top = importancias.head(15)[::-1]
plt.barh(top.index, top.values)
plt.title(f"Top 15 variáveis — {melhor_nome}")
plt.tight_layout()
plt.show()

## 9) Salvar o melhor modelo + gerar relatório (markdown)

Salvamos o modelo em `models/churn_model.joblib` e geramos `relatorio_parte2.md` com resultados.


In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)
model_path = "models/churn_model.joblib"
joblib.dump(melhor_pipe, model_path)
print("✅ Modelo salvo em:", model_path)

# Relatório simples
linhas = []
linhas.append("# Relatório — Telecom X (Parte 2: Previsão de Churn)")
linhas.append("")
linhas.append(f"Data: {pd.Timestamp.today().date()}")
linhas.append("")
linhas.append("## Modelos testados")
linhas.append("")
for nome, m in resultados.items():
    linhas.append(f"- **{nome}** — ROC-AUC: `{m['roc_auc']:.4f}` | PR-AUC: `{m['pr_auc']:.4f}`")
linhas.append("")
linhas.append("## Melhor modelo")
linhas.append("")
linhas.append(f"- **{melhor_nome}** (selecionado por ROC-AUC)")
linhas.append("")
linhas.append("## Top variáveis (permutation importance)")
linhas.append("")
for k, v in importancias.head(12).items():
    linhas.append(f"- `{k}`: `{v:.6f}`")
linhas.append("")
linhas.append("## Estratégias de retenção (para você adaptar)")
linhas.append("")
linhas.append("- Criar ações proativas para perfis com maior risco (ex.: contratos mensais, valores altos, pouco tempo de casa).")
linhas.append("- Oferecer upgrade/benefícios para segmentos críticos (plano, suporte, desconto temporário).")
linhas.append("- Campanhas específicas por método de pagamento/serviço, quando houver sinal claro nas variáveis.")
linhas.append("")
linhas.append("## Como reproduzir")
linhas.append("")
linhas.append("1. Rodar ETL (Parte 1): `python scripts/run_etl.py`")
linhas.append("2. Rodar este notebook e gerar `models/churn_model.joblib`")
linhas.append("")

with open("relatorio_parte2.md", "w", encoding="utf-8") as f:
    f.write("\n".join(linhas))

print("✅ Relatório salvo em: relatorio_parte2.md")

## 10) Inferência rápida (exemplo)

Exemplo de como prever churn para um conjunto de clientes (usando o modelo salvo).

In [ ]:
import joblib

modelo = joblib.load("models/churn_model.joblib")

# previsões para os 5 primeiros do teste
proba = modelo.predict_proba(X_test.iloc[:5])[:,1]
pred = (proba >= 0.5).astype(int)

saida = X_test.iloc[:5].copy()
saida["proba_churn"] = proba
saida["pred_churn"] = pred
display(saida)